In [2]:
import pandas as pd
import os
df = pd.read_csv("../0_data/0_raw/online_retail_II.csv")

print("Original Shape:", df.shape)

# 1. Remove Duplicates
df = df.drop_duplicates()
print("After Removing Duplicates:", df.shape)

# 2. Handle Cancellations and Invalid Quantities
# Remove invoices that start with 'C'
df = df[~df["Invoice"].astype(str).str.startswith("C")]
# Remove negative or zero quantities and zero prices
df = df[(df["Quantity"] > 0) & (df["Price"] > 0.0)]
print("After Removing Cancellations:", df.shape)

# 3. Handle Missing Customer IDs
df = df.dropna(subset=["Customer ID"])
print("After Removing Missing Customer IDs:", df.shape)

# 4. Change the InvoiceDate Column Type
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# 5. Create Revenue Column
df["Revenue"] = df["Quantity"] * df["Price"]
print(df.head())

# 6. Create an Invoice Month column
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')

# 7. Find the Cohort Month (the First Purchase Month) for Each Customer
df['CohortMonth'] = (
    df.groupby('Customer ID')['InvoiceMonth']
      .transform('min')
)

# 8. Calculate the Cohort Index (Months Passed since First Purchase)
df['CohortIndex'] = (
    (df['InvoiceMonth'].dt.year - df['CohortMonth'].dt.year) * 12
    + (df['InvoiceMonth'].dt.month - df['CohortMonth'].dt.month)
    + 1
)

# Save the cleaned dataset
os.makedirs("../0_data/1_cleaned", exist_ok=True)
df.to_csv("../0_data/1_cleaned/fact_transactions.csv", index=False)
print("Final Shape:", df.shape)

print("\n=== DATA CLEANING SUMMARY ===")
print(f"Rows Remaining: {len(df):,}")
print(f"Unique Customers: {df['Customer ID'].nunique():,}")
print(f"Unique Invoices: {df['Invoice'].nunique():,}")
print(f"Total Revenue (GBP): £{df['Revenue'].sum():,.2f}")

Original Shape: (1067371, 8)
After Removing Duplicates: (1033036, 8)
After Removing Cancellations: (1007913, 8)
After Removing Missing Customer IDs: (779425, 8)
  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

          InvoiceDate  Price  Customer ID         Country  Revenue  
0 2009-12-01 07:45:00   6.95      13085.0  United Kingdom     83.4  
1 2009-12-01 07:45:00   6.75      13085.0  United Kingdom     81.0  
2 2009-12-01 07:45:00   6.75      13085.0  United Kingdom     81.0  
3 2009-12-01 07:45:00   2.10      13085.0  United Kingdom    100.8  
4 2009-12-01 07:45:00   1.25      13085.0  United Kingdom     30.0  
Fina